# Train mô hình phát hiện vũ khí từ drone trên Google Colab

1. **Runtime → Change runtime type → T4 GPU** rồi bấm Save.
2. Chạy lần lượt từng ô (Shift + Enter).
3. Xong ô cuối, file `weapon_best.pt` được tải về máy: chép vào `models/weapon_best.pt` của dự án.

Kết quả được lưu cả vào Google Drive (`MyDrive/UAV_train`) để không mất khi Colab ngắt kết nối.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/tuankiet4725/CNPM.git UAV
%cd UAV
!pip install -q ultralytics roboflow

Nhập API key Roboflow (https://app.roboflow.com/settings/api). Key chỉ nằm trong phiên Colab này, không lưu vào notebook.

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')

Tải và gộp dataset (bộ drone 4K được cắt thành ô 1280 px). Mất khoảng 5–10 phút.

In [ ]:
!python training/prepare_roboflow.py

Gắn Google Drive để lưu kết quả train (Colab có thể ngắt sau vài giờ).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/UAV_train'
os.makedirs(SAVE, exist_ok=True)

Train trên GPU: `yolov8s` (chính xác hơn `yolov8n`), ảnh 640, tối đa 60 epoch, tự dừng nếu 20 epoch không cải thiện.

Nếu báo hết bộ nhớ GPU (CUDA out of memory), giảm `--batch` xuống 16.

In [ ]:
!python training/train.py --data datasets/weapons_uav/data.yaml --model yolov8s.pt --imgsz 640 --epochs 60 --batch 32 --device 0
!cp -r runs/detect/runs/weapon "$SAVE/" && cp models/weapon_best.pt "$SAVE/"

Nếu Colab ngắt giữa chừng: chạy lại các ô từ đầu đến ô gắn Drive (bỏ qua ô train), rồi chạy ô dưới để train tiếp từ bản lưu trên Drive.

In [ ]:
!mkdir -p runs/detect/runs && cp -r "$SAVE/weapon" runs/detect/runs/
!python training/train.py --resume runs/detect/runs/weapon/train/weights/last.pt
!cp -r runs/detect/runs/weapon "$SAVE/" && cp models/weapon_best.pt "$SAVE/"

Xem biểu đồ kết quả và tải mô hình về máy.

In [ ]:
from IPython.display import Image, display
display(Image('runs/detect/runs/weapon/train/results.png', width=900))
from google.colab import files
files.download('models/weapon_best.pt')